# **Hypothesis Testing**

**Treatment Group & Control Group**

Treatment Group (New Webpage): Users in this group will be exposed to the new webpage design. The effectiveness of the new webpage design will be measured by comparing the conversion rates of users (who actually make purchase of the company's products after visiting this new webpage) in this group to those in the control groups.

Control Group 1 (Placebo): Users in this group will be presented with an identical-looking webpage that serves as a placebo. This group represents the baseline scenario where users are exposed to the current webpage design without any changes. It means that in Control Group 1, users will see a webpage that looks exactly like the current one (new one) but doesn't have any actual changes. This group helps us understand how users typically behave on the current webpage without any alterations. It's like giving users a fake version of the webpage to see how they respond, so we can compare their behavior to those users who see the real changes in the actual new webpage.

Control Group 2 (Old Webpage): Users in this group will be shown a webpage that is already in use and has demonstrated effectiveness in terms of conversion rates. It means that users in Control Group 2 will see the same old webpage that is currently being used. This webpage has been proven to be effective in terms of converting visitors into purchasers (or customers) in the past.

-- This group serves as a benchmark to evaluate whether the new webpage design outperforms the existing treatment.

-- This group (Control Group 2) acts as a standard for comparison to see if the new webpage design performs better than the current one. We will use the conversion rates observed in Control Group 2 to assess whether the changes made in the new webpage design lead to better results or not.

Note:

(1) In the dataset there are no users from Control Group 1 (Placebo). The provided Placebo description is only for context.

(2) In the Mini Project, users from Control Group 2 (Old Webpage) will be used as the 'Control Group'.

In [4]:
# Importing required packages
import pandas as pd
import numpy as np
import scipy.stats as ss
import statsmodels.api as sm
import math as mt
import itertools
import random
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
from scipy.stats import norm
%matplotlib inline

In [7]:
# Load the dataset
data = pd.read_csv('ab_data.csv')
df = data.copy()
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [9]:
df['group'].value_counts()

group
treatment    147276
control      147202
Name: count, dtype: int64

In [14]:
print("The number of rows in the dataset:", df.shape[0])

The number of rows in the dataset: 294478


# Data Cleaning

In [15]:
# The number of unique users in the dataset
df.user_id.nunique()

290584

In [16]:
# Conversion Rate
df.query('converted == 1')['converted'].count() / df.shape[0] 

np.float64(0.11965919355605512)

In [19]:
# Identify how mant users in treatment group did not match with new_page
N1 = df.query('group == "treatment" and landing_page != "new_page"').shape[0]
N1

1965

In [22]:
# Identify how mant users in control group did not match with old_page
N2 = df.query('group == "control" and landing_page != "old_page"').shape[0]
N2

1928

In [23]:
# Total number of non-line up
N = N1 + N2
N

3893

In [29]:
# Check for any missing values
df.isnull().sum().sum()

np.int64(0)

In [34]:
# Check datatype of each column
df.dtypes

user_id          int64
timestamp       object
group           object
landing_page    object
converted        int64
dtype: object

# Identify the not aligned rows

With the above dataset the requirement is to first identify the rows in that dataset where the treatment group is aligned with the new_page and where the control group is aligned with the old_page.

In [ ]:
# Treatment is aligned with new_page or control is aligned with old_page
df2 = df.iloc[df.query('group == "treatment" and landing_page == "new_page"').index.values]

df3 = df.iloc[df.query('group == "control" and landing_page == "old_page"').index.values]

In [ ]:
df2 = pd.concat([df2, df3], ignore_index=False)
df2

,user_id,timestamp,group,landing_page,converted
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
6,679687,2017-01-19 03:26:46.940749,treatment,new_page,1
8,817355,2017-01-04 17:58:08.979471,treatment,new_page,1
9,839785,2017-01-15 18:11:06.610965,treatment,new_page,1
...,...,...,...,...,...
294471,718310,2017-01-21 22:44:20.378320,control,old_page,0
294473,751197,2017-01-03 22:28:38.630509,control,old_page,0
294474,945152,2017-01-12 00:51:57.078372,control,old_page,0
294475,734608,2017-01-22 11:45:03.439544,control,old_page,0


In [ ]:
df2.shap

,user_id,timestamp,group,landing_page,converted
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
6,679687,2017-01-19 03:26:46.940749,treatment,new_page,1
8,817355,2017-01-04 17:58:08.979471,treatment,new_page,1
9,839785,2017-01-15 18:11:06.610965,treatment,new_page,1
...,...,...,...,...,...
294471,718310,2017-01-21 22:44:20.378320,control,old_page,0
294473,751197,2017-01-03 22:28:38.630509,control,old_page,0
294474,945152,2017-01-12 00:51:57.078372,control,old_page,0
294475,734608,2017-01-22 11:45:03.439544,control,old_page,0


Now, with the help of the new dataset, we need to identify the misaligned rows in the dataset where treatment is not aligned with new_page or control is not aligned with old_page

This can be done by checking the values 'treatment' and 'control' under the 'group' column to ensure they do not correspond with the values 'new_page' and 'old_page' under the 'landing_page' column, respectively.

For the rows where treatment is not aligned with new_page or control is not aligned with old_page, we cannot be sure if this row truly received the new or old page.

In [39]:
# Identify misaligned rows where treatment is not aligned with new_page or control is not aligned with old_page
df_misaligned = df[~df.index.isin(df2.index)]
df_misaligned

,user_id,timestamp,group,landing_page,converted
22,767017,2017-01-12 22:58:14.991443,control,new_page,0
240,733976,2017-01-11 15:11:16.407599,control,new_page,0
308,857184,2017-01-20 07:34:59.832626,treatment,old_page,0
327,686623,2017-01-09 14:26:40.734775,treatment,old_page,0
357,856078,2017-01-12 12:29:30.354835,treatment,old_page,0
...,...,...,...,...,...
294014,813406,2017-01-09 06:25:33.223301,treatment,old_page,0
294200,928506,2017-01-13 21:32:10.491309,control,new_page,0
294252,892498,2017-01-22 01:11:10.463211,treatment,old_page,0
294253,886135,2017-01-06 12:49:20.509403,control,new_page,0


In [ ]:
# Double Check all of the correct rows were removed - this should be 0
df2[((df2['group'] == 'treatment') == (df2['landing_page'] == 'new_page')) == False].shape[0]

0